# Step 1: 载入并预处理实验 JSON 数据
使用 `pandas` 和 `json` 库读取实验日志。将嵌套的 `result` 和 `config` 字典平铺展开为 DataFrame，以便后续进行数据透视操作。

In [ ]:
# 检查依赖并安装 (如果需要)
# !pip install pandas matplotlib numpy

In [ ]:
import pandas as pd
import json
import glob
import os
from pathlib import Path

candidate_dirs = [Path('experiment_results'), Path('../experiment_results'), Path('.')]
results_dir = next((path for path in candidate_dirs if path.exists()), Path('experiment_results')) 
json_files = None

# 寻找最新的 experiment_summary_*.json 文件
json_files = (
    glob.glob(str(results_dir / "experiment_summary_*.json"))
    if json_files is None
    else json_files
)
if not json_files:
    print("找不到任何实验总结 JSON 文件")
    # 如果没有自动生成的，尝试手动指定一个
    filename = str(results_dir / "experiment_summary_20260201_121422.json")
else:
    # 获取最新的文件
    filename = max(json_files, key=os.path.getmtime)

print(f"正在读取文件: {filename}")

with open(filename, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 将嵌套的 JSON 数据平铺展开
df = pd.json_normalize(data)
df.head()

# Step 2: 筛选 Adaptive Tridecoding 模式数据
根据 `config.eval_mode` 或 `result.eval_mode` 字段筛选出 Adaptive Tridecoding 模式的记录。根据提供的 outline 要求，我们主要关注 `adaptive_tridecoding` 模式。如果数据中使用 `large` 表示该模式，也会一并考虑。

In [ ]:
# 筛选 adaptive_tridecoding
# 注意：根据需求，我们只针对 "adaptive_tridecoding" 模式。
# 如果您的数据中该模式被标记为 "large"，请将 "large" 加入 target_target_modes。
target_modes = ['adaptive_tridecoding']

# 优先在 result 或 config 中寻找 eval_mode
mode_col = None
for col in ['result.eval_mode', 'config.eval_mode']:
    if col in df.columns:
        mode_col = col
        break

if mode_col:
    filtered_df = df[df[mode_col].isin(target_modes)].copy()
else:
    # 模糊匹配
    mode_cols = [c for c in df.columns if 'eval_mode' in c]
    if mode_cols:
        filtered_df = df[df[mode_cols[0]].isin(target_modes)].copy()
    else:
        filtered_df = df.copy()

print(f"筛选出的 {target_modes} 记录数: {len(filtered_df)}")

# Step 3: 解析时间组成指标并进行数据格式转换
提取 `computation_time`, `communication_time`, `queuing_time` 以及其他开销时间。计算各分量占 `wall_time` 的比例。同时提取数据集名称和模型名称。

In [ ]:
import numpy as np

# 定义我们感兴趣的时间列
time_cols = {
    'result.computation_time': 'Computation',
    'result.communication_time': 'Communication',
    'result.queuing_time': 'Queuing',
    'result.arp_overhead_time': 'ARP Overhead',
    'result.dra_overhead_time': 'DRA Overhead'
}

# 确保列存在，不存在则补0
for col in time_cols.keys():
    if col not in filtered_df.columns:
        filtered_df[col] = 0.0

# 提取并重命名数据集
def get_dataset_name(x):
    if pd.isna(x): return 'unknown'
    name = x.split('/')[-1].replace('eval_', '').replace('.py', '')
    # 重命名映射
    mapping = {
        'mt_bench': 'MT-Bench',
        'mt_bench_eval': 'MT-Bench',
        'mt_bench_noeval': 'MT-Bench',
        'mt_bench_nooeval': 'MT-Bench',
        'gsm8k': 'GSM8K',
        'humaneval': 'HumanEval'
    }
    return mapping.get(name, name)

filtered_df['dataset'] = filtered_df['config.eval_dataset'].apply(get_dataset_name)

# 统一模型名称 (只保留最后一部分)
def simplify_model_name(name):
    if isinstance(name, str) and "/" in name:
        return name.split("/")[-1]
    return name

filtered_df['model'] = filtered_df['result.target_model'].apply(simplify_model_name).fillna('unknown')

# 计算各组分的占比
filtered_df['total_measured_time'] = filtered_df[list(time_cols.keys())].sum(axis=1)

for col, label in time_cols.items():
    # 同时保留绝对值列和占比列
    filtered_df[f'{label}_abs'] = filtered_df[col]
    filtered_df[f'{label}_pct'] = filtered_df[col] / filtered_df['total_measured_time'].replace(0, 1) * 100

# 准备列名单
abs_cols = [f'{label}_abs' for label in time_cols.values()]
pct_cols = [f'{label}_pct' for label in time_cols.values()]

final_df = filtered_df[['model', 'dataset', 'result.wall_time', 'total_measured_time'] + abs_cols + pct_cols].copy()
final_df.head()

# Step 4: 生成单图分组堆积柱状图
使用 `matplotlib` 绘制单个坐标系中的分组堆积柱状图。横轴按数据集分组，组内展示不同模型，颜色表示时间构成。

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# 1. 聚合数据: 按模型和数据集聚合，不进行跨数据集合并
agg_df = final_df.groupby(['model', 'dataset']).mean(numeric_only=True).reset_index()

# 2. 输出原始数据表格 (包含数据集维度)
display(Markdown("### Wall Time Composition Breakdown by Model and Dataset"))
display_cols = ['model', 'dataset', 'result.wall_time'] + abs_cols + pct_cols
table_df = agg_df[display_cols].round(2)
display(table_df)

# 3. 绘制单图分组堆积柱状图
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 12,
})

dataset_order = ['GSM8K', 'HumanEval', 'MT-Bench']
model_order = ['Llama-2', 'Qwen1.5', 'Qwen3']
datasets = [d for d in dataset_order if d in agg_df['dataset'].unique()]
datasets += [d for d in agg_df['dataset'].unique() if d not in datasets]
models = [m for m in model_order if m in agg_df['model'].unique()]
models += [m for m in agg_df['model'].unique() if m not in models]

component_labels = [col.replace('_pct', '') for col in pct_cols]
component_colors = {
    'Computation': '#1f77b4',
    'Communication': '#ff7f0e',
    'Queuing': '#2ca02c',
    'ARP Overhead': '#d62728',
    'DRA Overhead': '#9467bd',
}
model_display_names = {
    'Llama-2': 'Llama-2',
    'Qwen1.5': 'Qwen-1.5',
    'Qwen3': 'Qwen-3',
}

def get_model_display_name(model):
    model = str(model)
    if 'Llama-2' in model:
        return 'Llama-2'
    if 'Qwen1.5' in model or 'Qwen-1.5' in model:
        return 'Qwen-1.5'
    if 'Qwen3' in model or 'Qwen-3' in model:
        return 'Qwen-3'
    return model_display_names.get(model, model)

bar_width = 0.28
bar_gap = 0.07
group_gap = 0.85
group_width = len(models) * bar_width
group_starts = np.arange(len(datasets)) * (group_width + (len(models) - 1) * bar_gap + group_gap)

fig, ax = plt.subplots(figsize=(7.2, 3.1))

for dataset_idx, dataset in enumerate(datasets):
    group_start = group_starts[dataset_idx]
    for model_idx, model in enumerate(models):
        x = group_start + model_idx * (bar_width + bar_gap)
        row = agg_df[(agg_df['dataset'] == dataset) & (agg_df['model'] == model)]
        if row.empty:
            continue

        bottom = 0
        for col, label in zip(pct_cols, component_labels):
            value = float(row.iloc[0][col])
            ax.bar(
                x,
                value,
                width=bar_width,
                bottom=bottom,
                color=component_colors.get(label),
                edgecolor='white',
                linewidth=0.45,
                label=label if dataset_idx == 0 and model_idx == 0 else None,
            )
            bottom += value

# 模型作为底部小刻度标签，数据集作为组标签，避免三个子图重复坐标轴。
bar_positions = []
bar_labels = []
for dataset_idx, dataset in enumerate(datasets):
    for model_idx, model in enumerate(models):
        bar_positions.append(group_starts[dataset_idx] + model_idx * (bar_width + bar_gap))
        bar_labels.append(get_model_display_name(model))

ax.set_xticks(bar_positions)
ax.set_xticklabels(bar_labels, rotation=28, ha='right', rotation_mode='anchor')
ax.tick_params(axis='x', length=0, pad=4)

for dataset_idx, dataset in enumerate(datasets):
    center = group_starts[dataset_idx] + (len(models) - 1) * (bar_width + bar_gap) / 2
    ax.text(center, 103.5, dataset, ha='center', va='bottom', fontsize=11)
    if dataset_idx > 0:
        sep_x = group_starts[dataset_idx] - group_gap / 2
        ax.axvline(sep_x, color='#d0d0d0', linewidth=0.8, ymin=0.05, ymax=0.95)

ax.set_ylabel('Percentage of Time (%)')
ax.set_ylim(0, 108)
ax.set_yticks(np.arange(0, 101, 20))
ax.yaxis.grid(True, linestyle='-', color='#e6e6e6', linewidth=0.8)
ax.set_axisbelow(True)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.spines['left'].set_color('#bbbbbb')
ax.spines['bottom'].set_color('#bbbbbb')

legend = ax.legend(
    title='Time Components',
    loc='center left',
    bbox_to_anchor=(1.01, 0.5),
    frameon=False,
    borderaxespad=0,
)
legend.get_title().set_fontsize(13)

fig.subplots_adjust(right=0.77, top=0.86, bottom=0.28, left=0.09)

output_path = 'walltime_composition_grouped.pdf'
plt.savefig(output_path, bbox_inches='tight', dpi=300)
print(f'图表已保存至: {output_path}')

plt.show()


# Step 5: 按模型聚合后的平均时间组成饼图
由于不同数据集之间的时间组成差异较小，这里对每个模型在所有数据集上的时间组成百分比取均值，并使用三个饼图展示。

In [ ]:
# 按模型跨数据集取均值，生成更紧凑的 latency composition 饼图
pie_df = agg_df.copy()
pie_df['model_display'] = pie_df['model'].apply(get_model_display_name)
pie_df = pie_df.groupby('model_display')[pct_cols].mean().reset_index()

pie_model_order = ['Llama-2', 'Qwen-1.5', 'Qwen-3']
pie_models = [m for m in pie_model_order if m in pie_df['model_display'].unique()]
pie_models += [m for m in pie_df['model_display'].unique() if m not in pie_models]

display(Markdown('### Average Wall Time Composition by Model'))
display(pie_df.set_index('model_display').loc[pie_models].round(2))

fig, axes = plt.subplots(1, len(pie_models), figsize=(7.6, 2.7))
if len(pie_models) == 1:
    axes = [axes]

pie_colors = [component_colors[label] for label in component_labels]
pie_model_labels = {
    'Llama-2': 'Llama-2-13B',
    'Qwen-1.5': 'Qwen-1.5-7B-Chat',
    'Qwen-3': 'Qwen-3-14B',
}

def autopct_fmt(value):
    return f'{value:.0f}%' if value >= 4 else ''

for ax, model in zip(axes, pie_models):
    values = pie_df.loc[pie_df['model_display'] == model, pct_cols].iloc[0].to_numpy(dtype=float)
    values = values / values.sum() * 100
    wedges, texts, autotexts = ax.pie(
        values,
        colors=pie_colors,
        startangle=90,
        counterclock=False,
        autopct=autopct_fmt,
        pctdistance=0.68,
        wedgeprops={'linewidth': 0.7, 'edgecolor': 'white'},
        textprops={'fontsize': 13},
    )
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(13)
    ax.text(
        0.5,
        -0.12,
        pie_model_labels.get(model, model),
        transform=ax.transAxes,
        ha='center',
        va='top',
        fontsize=14,
        fontweight='bold',
    )
    ax.set_aspect('equal')

fig.legend(
    wedges,
    component_labels,
    title='Time Components',
    loc='center left',
    bbox_to_anchor=(0.88, 0.5),
    frameon=False,
    fontsize=14,
    title_fontsize=16,
)
fig.subplots_adjust(left=0.03, right=0.82, top=0.92, bottom=0.26, wspace=0.12)

output_path = 'walltime_composition_model_mean_pies.pdf'
plt.savefig(output_path, bbox_inches='tight', dpi=300)
print(f'图表已保存至: {output_path}')

plt.show()
